# D.R.O.N.A. — Notebook 07: ACT Gesture Policy (LeRobot, Colab T4)

**Research Contribution C3** — train an *Action Chunking with Transformers* (ACT; Zhao et al. 2023) policy on D.R.O.N.A. gesture demonstrations using the official **HuggingFace LeRobot** trainer, and compare it against the keyframe baseline on smoothness (jerk).

Pipeline: keyframe seed demos → `LeRobotDataset` (v2) → LeRobot ACT training → checkpoint → `drona.interaction.sim_eval` comparison.

**Runtime:** Colab **T4 GPU** (`Runtime → Change runtime type → T4 GPU`). ~10–20 min for all gestures.

> The repo runs fully without LeRobot via the `KeyframePolicy` fallback. This notebook is the *training* path that produces the ACT checkpoints `PolicyRouter` loads automatically.

## 1. Environment check

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "No GPU — set Runtime to T4 GPU")

## 2. Install LeRobot + clone D.R.O.N.A.

In [ ]:
%pip install -q "git+https://github.com/huggingface/lerobot.git"
import os
if not os.path.exists("D.R.O.N.A"):
    # Replace with your repo URL, or upload the project folder to Colab.
    !git clone https://github.com/<your-username>/D.R.O.N.A.git 2>/dev/null || echo "Clone manually or upload the repo, then set the path below."
REPO = "D.R.O.N.A" if os.path.exists("D.R.O.N.A") else "."
%cd $REPO
%pip install -q -e . || %pip install -q numpy loguru pydantic

## 3. Generate seed demonstrations

We bootstrap from the pre-programmed keyframe gestures, adding small Gaussian jitter per episode so ACT sees variation (real teleop demos would replace these). All frames are clamped to joint limits.

In [ ]:
import numpy as np
from drona.interaction.demonstration import (
    GESTURE_KEYFRAMES, DemonstrationDataset, DemonstrationEpisode,
    interpolate_keyframes, clamp_joints,
)

EPISODES_PER_GESTURE = 25
JITTER = 0.02  # radians
rng = np.random.default_rng(42)

ds = DemonstrationDataset(name="drona_gestures")
ep_idx = 0
for gesture, keyframes in GESTURE_KEYFRAMES.items():
    for _ in range(EPISODES_PER_GESTURE):
        traj = interpolate_keyframes(keyframes, dt=0.05)
        ep = DemonstrationEpisode(episode_index=ep_idx, gesture_label=gesture,
                                  metadata={"source": "keyframe+jitter"})
        for i, (q, t) in enumerate(traj):
            nq = traj[i + 1][0] if i + 1 < len(traj) else q
            obs = clamp_joints(q + rng.normal(0, JITTER, size=q.shape).astype(np.float32))
            act = clamp_joints(nq + rng.normal(0, JITTER, size=nq.shape).astype(np.float32))
            ep.add_frame(obs=obs, action=act, timestamp=t)
        ep.mark_terminal()
        ds.add_episode(ep)
        ep_idx += 1

print(f"{len(ds.episodes)} episodes, {ds.total_frames} frames")
print("gesture counts:", ds.gesture_counts)

## 4. Convert to a `LeRobotDataset`

In [ ]:
from drona.interaction.lerobot_dataset import build_lerobot_dataset, LEROBOT_FEATURES
print("features:", LEROBOT_FEATURES)

lerobot_ds = build_lerobot_dataset(ds, repo_id="drona/gestures", fps=20,
                                   root="data/lerobot/drona_gestures")
print("LeRobotDataset frames:", lerobot_ds.num_frames if hasattr(lerobot_ds, 'num_frames') else 'created')

## 5. Train ACT

We use LeRobot's `train` entrypoint via its Python API. Hyperparameters are sized for the small gesture dataset on a T4. If the LeRobot CLI is preferred, the equivalent shell command is shown in the next cell.

In [ ]:
# Option A — LeRobot training CLI (recommended; API names drift across versions).
!python -m lerobot.scripts.train \
    --dataset.repo_id=drona/gestures \
    --dataset.root=data/lerobot/drona_gestures \
    --policy.type=act \
    --policy.chunk_size=10 \
    --policy.n_action_steps=10 \
    --policy.dim_model=256 \
    --batch_size=32 \
    --steps=20000 \
    --output_dir=data/checkpoints/act \
    --job_name=drona_act \
    --device=cuda \
    --wandb.enable=false

> **Per-gesture checkpoints:** `PolicyRouter` loads `data/checkpoints/<gesture>/`. To train one policy per gesture instead of a single conditioned policy, loop over `GESTURE_KEYFRAMES`, build a per-gesture `LeRobotDataset`, and set `--output_dir=data/checkpoints/<gesture>`. The repo's `scripts/train_act.py` also implements a self-contained per-gesture loop without the LeRobot CLI.

## 6. Evaluate: ACT vs keyframe baseline (jerk / success)

In [ ]:
import json
from drona.interaction.act_policy import KeyframePolicy, LeRobotACTPolicy
from drona.interaction.sim_eval import compare_policies

CKPT = "data/checkpoints/act"  # adjust to per-gesture dir if used

def act_factory(gesture):
    try:
        return LeRobotACTPolicy(f"{CKPT}/{gesture}", device="cuda")
    except Exception:
        return LeRobotACTPolicy(CKPT, device="cuda")

result = compare_policies(
    base_factory=lambda g: KeyframePolicy(g),
    other_factory=act_factory,
)
print(json.dumps(result["delta"], indent=2))
print("ACT smoother than keyframe:" , result["smoother"])

## 7. Save checkpoint to Drive

Copy `data/checkpoints/` back into the repo (or mount Drive) so `PolicyRouter` picks up the ACT policy on the local/robot machine. Record the jerk delta in `docs/research_papers.md` for the viva.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r data/checkpoints /content/drive/MyDrive/drona_checkpoints
print("Done — checkpoints in data/checkpoints/")